In [2]:
pip install Pillow numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install Pillow


[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: C:\Users\isabe\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import os
import sys
import argparse
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
import numpy as np
from PIL import Image


In [4]:
class ImageCollageGenerator:
    """Generate collages from hierarchical folder structures."""
    
    SUPPORTED_FORMATS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}
    
    def __init__(self, 
                 input_dir: str,
                 output_dir: Optional[str] = None,
                 max_images_per_collage: int = 20,
                 collage_size: Tuple[int, int] = (1920, 1080),
                 padding: int = 10,
                 background_color: str = 'white'):
        
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir) if output_dir else self.input_dir / 'collages'
        self.max_images = max_images_per_collage
        self.collage_size = collage_size
        self.padding = padding
        self.background_color = background_color
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.folder_levels: Dict[int, List[Path]] = defaultdict(list)
        self.folder_images: Dict[Path, List[Path]] = defaultdict(list)
        
    def find_valid_images(self, directory: Path) -> List[Path]:
        """Find valid image files, filtering out system files and invalid images"""
        all_files = []
        for ext in self.SUPPORTED_FORMATS:
            all_files.extend(directory.glob(f'*{ext}'))
            all_files.extend(directory.glob(f'*{ext.upper()}'))
        
        # Sort files
        all_files = sorted(all_files)
        
        # Filter out system files (starting with ._ or other invalid patterns)
        valid_images = []
        invalid_count = 0
        
        for file_path in all_files:
            # Skip system files and hidden files
            if file_path.name.startswith('._') or file_path.name.startswith('~'):
                invalid_count += 1
                continue
            
            # Try to verify it's actually an image file
            try:
                with Image.open(file_path) as img:
                    img.verify()  # Verify it's a valid image
                valid_images.append(file_path)
            except Exception as e:
                print(f"    Skipping invalid image: {file_path.name} - {e}")
                invalid_count += 1
                continue
        
        if invalid_count > 0:
            print(f"    Filtered out {invalid_count} invalid files")
        
        return valid_images
    
    def calculate_grid_layout(self, num_images: int) -> Tuple[int, int]:
        if num_images <= 0:
            return (1, 1)
        
        # Simple grid calculation
        cols = int(np.ceil(np.sqrt(num_images)))
        rows = int(np.ceil(num_images / cols))
        return (rows, cols)
    
    def create_collage(self, images: List[Path], output_path: Path) -> bool:
        """Create a collage from valid images"""
        if not images:
            print(f"  No valid images found")
            return False
        
        # Use the LAST images instead of first (to avoid system files)
        if len(images) > self.max_images:
            images_to_use = images[-self.max_images:]  # Take from the end
            print(f"  Using last {self.max_images} images (to avoid system files)")
        else:
            images_to_use = images
        
        num_images = len(images_to_use)
        
        print(f"  Creating collage with {num_images} valid images")
        print(f"  Collage size: {self.collage_size}")
        
        # Calculate grid layout
        rows, cols = self.calculate_grid_layout(num_images)
        print(f"  Grid: {rows}x{cols}")
        
        # Calculate cell size
        cell_width = (self.collage_size[0] - (self.padding * (cols + 1))) // cols
        cell_height = (self.collage_size[1] - (self.padding * (rows + 1))) // rows
        
        print(f"  Cell size: {cell_width}x{cell_height}")
        
        # Create canvas
        if self.background_color == 'white':
            canvas = Image.new('RGB', self.collage_size, (255, 255, 255))
        else:
            canvas = Image.new('RGB', self.collage_size, (0, 0, 0))
        
        images_placed = 0
        
        for idx, img_path in enumerate(images_to_use):
            try:
                print(f"    Processing: {img_path.name}")
                
                # Open image
                img = Image.open(img_path)
                
                # Convert to RGB if necessary
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                # Resize maintaining aspect ratio
                img_resized = img.copy()
                img_resized.thumbnail((cell_width, cell_height), Image.Resampling.LANCZOS)
                
                print(f"      Original: {img.size}, Resized: {img_resized.size}")
                
                # Calculate position
                row = idx // cols
                col = idx % cols
                
                x_pos = self.padding + col * (cell_width + self.padding)
                y_pos = self.padding + row * (cell_height + self.padding)
                
                # Center in cell
                x_offset = (cell_width - img_resized.width) // 2
                y_offset = (cell_height - img_resized.height) // 2
                
                final_x = x_pos + x_offset
                final_y = y_pos + y_offset
                
                print(f"      Position: ({final_x}, {final_y})")
                
                # Paste image
                canvas.paste(img_resized, (final_x, final_y))
                images_placed += 1
                
                img.close()  # Close the image file
                
            except Exception as e:
                print(f"    Failed to process {img_path.name}: {e}")
                continue
        
        # Save collage
        try:
            canvas.save(output_path, quality=95, optimize=True)
            print(f"  ✅ SUCCESS: Created {output_path.name} with {images_placed} images")
            return True
        except Exception as e:
            print(f"  ❌ FAILED to save {output_path}: {e}")
            return False
    
    def analyze_folder_structure(self) -> None:
        print(f"Analyzing folder structure in: {self.input_dir}")
        
        for root, dirs, files in os.walk(self.input_dir):
            root_path = Path(root)
            
            if self.output_dir in root_path.parents or root_path == self.output_dir:
                continue
            
            # Use our new function that filters invalid images
            images = self.find_valid_images(root_path)
            
            if images:
                try:
                    depth = len(root_path.relative_to(self.input_dir).parts)
                except ValueError:
                    depth = 0
                
                self.folder_levels[depth].append(root_path)
                self.folder_images[root_path] = images
                
                print(f"  Found {len(images)} valid images in {root_path.relative_to(self.input_dir)} (level {depth})")
    
    def process_folders(self) -> None:
        if not self.folder_levels:
            print("No folders with valid images found!")
            return
        
        max_depth = max(self.folder_levels.keys())
        
        print(f"\nProcessing {len(self.folder_levels)} levels, from depth {max_depth} to 0")
        print(f"Total folders to process: {sum(len(folders) for folders in self.folder_levels.values())}")
        
        for depth in range(max_depth, -1, -1):
            if depth not in self.folder_levels:
                continue
            
            folders = self.folder_levels[depth]
            print(f"\n--- Processing level {depth} ({len(folders)} folders) ---")
            
            for folder_path in folders:
                images = self.folder_images.get(folder_path, [])
                if not images:
                    continue
                
                relative_path = folder_path.relative_to(self.input_dir)
                safe_name = '_'.join(relative_path.parts).replace(' ', '_').replace('/', '_')
                output_filename = f"collage_{safe_name}.png"
                output_path = self.output_dir / output_filename
                
                print(f"Processing: {relative_path}")
                self.create_collage(images, output_path)
    
    def run(self) -> None:
        print("=" * 60)
        print("Image Collage Generator - Fixed Version")
        print("=" * 60)
        print(f"Input directory: {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Max images per collage: {self.max_images}")
        print(f"Collage size: {self.collage_size[0]}x{self.collage_size[1]}")
        print("=" * 60)
        
        self.analyze_folder_structure()
        self.process_folders()
        
        print("\n" + "=" * 60)
        print("Collage generation complete!")
        print(f"Collages saved to: {self.output_dir}")
        print("=" * 60)


# Test function that uses the LAST images instead of first
def test_with_valid_images():
    """Test with valid images from the end of the list"""
    test_generator = ImageCollageGenerator(
        input_dir="C:\\Users\\isabe\\Unity\\artbench-10-imagefolder",
        output_dir="C:\\Users\\isabe\\Unity\\collages_fixed",
        max_images_per_collage=4,  # Small test
        collage_size=(1920, 1080),
        padding=20,
        background_color='white'
    )
    
    # Test with one folder
    test_folder = Path("C:\\Users\\isabe\\Unity\\artbench-10-imagefolder") / "art_nouveau"
    images = test_generator.find_valid_images(test_folder)
    
    if images:
        print(f"Found {len(images)} valid images")
        print(f"Using last {min(4, len(images))} images:")
        for img in images[-4:]:
            print(f"  - {img.name}")
        
        output_path = test_generator.output_dir / "test_collage_fixed.png"
        test_generator.create_collage(images, output_path)
    else:
        print("No valid images found!")


# Main function for notebook use
def create_collages_from_folder(input_folder: str, output_folder: str = None, **kwargs):
    generator = ImageCollageGenerator(
        input_dir=input_folder,
        output_dir=output_folder,
        **kwargs
    )
    generator.run()
    return generator


print("✅ Fixed collage generator ready!")
print("\nKey improvements:")
print("1. Filters out system files (._ files)")
print("2. Uses LAST images instead of first")
print("3. Verifies images are actually valid")
print("4. Better debugging output")

print("\nRun the test first:")
print("test_with_valid_images()")

print("\nThen run the full version:")
print("""
create_collages_from_folder(
    input_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\artbench-10-imagefolder",
    output_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\collages_final",
    max_images_per_collage=12,
    collage_size=(1920, 1080),
    padding=20
)
""")

✅ Fixed collage generator ready!

Key improvements:
1. Filters out system files (._ files)
2. Uses LAST images instead of first
3. Verifies images are actually valid
4. Better debugging output

Run the test first:
test_with_valid_images()

Then run the full version:

create_collages_from_folder(
    input_folder="C:\\Users\\isabe\\Unity\\artbench-10-imagefolder",
    output_folder="C:\\Users\\isabe\\Unity\\collages_final",
    max_images_per_collage=12,
    collage_size=(1920, 1080),
    padding=20
)



In [5]:
# Run this test first to debug the issue
# Test with the fixed version
test_with_valid_images()

No valid images found!


In [20]:
# Example usage in notebook format
def create_collage_example():
    """Example of how to use the collage generator in a notebook."""
    
    # Define your input and output paths
    input_folder = "C:/Users/isabe/Unity/artbench-10-imagefolder"  # Change this to your folder path
    output_folder = "C:/Users/isabe/Unity/xollages"  # Change this to your desired output
    
    # Create the generator
    generator = ImageCollageGenerator(
        input_dir=input_folder,
        output_dir=output_folder,
        max_images_per_collage=12,  # Adjust as needed
        collage_size=(1920, 1080),   # Adjust as needed
        padding=15,
        background_color='black'
    )
    
    # Run the collage generation
    generator.run()
    
    return generator

# Uncomment and run the following line with your actual paths:
create_collage_example()

Image Collage Generator - Fixed Version
Input directory: C:\Users\isabe\Unity\artbench-10-imagefolder
Output directory: C:\Users\isabe\Unity\xollages
Max images per collage: 12
Collage size: 1920x1080
Analyzing folder structure in: C:\Users\isabe\Unity\artbench-10-imagefolder
    Filtered out 12000 invalid files
  Found 12000 valid images in art_nouveau (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in baroque (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in expressionism (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in impressionism (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in post_impressionism (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in realism (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in renaissance (level 1)
    Filtered out 12000 invalid files
  Found 12000 valid images in romanticism (level 1)
    Filt

In [6]:
#!/usr/bin/env python3
"""
Fixed Grid Collage Generator - 5x4 grid with black background + Image Copy
"""

import os
import sys
import shutil
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
from PIL import Image
import numpy as np


class FixedGridCollageGenerator:
    """Generate collages with fixed 5x4 grid and black background + copy source images."""
    
    SUPPORTED_FORMATS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}
    
    def __init__(self, 
                 input_dir: str,
                 output_dir: Optional[str] = None,
                 images_per_collage: int = 20,  # Fixed to 20 for 5x4 grid
                 collages_per_folder: int = 20,
                 collage_size: Tuple[int, int] = (1920, 1080),
                 padding: int = 5):
        
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir) if output_dir else self.input_dir / 'collages_fixed_grid'
        self.images_output_dir = Path(output_dir).parent / 'collage_images' if output_dir else self.input_dir / 'collage_images'
        self.images_per_collage = 20  # Force 20 images for 5x4 grid
        self.collages_per_folder = collages_per_folder
        self.collage_size = collage_size
        self.padding = padding
        
        # Create both output directories
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.images_output_dir.mkdir(parents=True, exist_ok=True)
        
        self.folder_levels: Dict[int, List[Path]] = defaultdict(list)
        self.folder_images: Dict[Path, List[Path]] = defaultdict(list)
        
    def find_valid_images(self, directory: Path) -> List[Path]:
        """Find valid image files, filtering out system files and invalid images"""
        all_files = []
        for ext in self.SUPPORTED_FORMATS:
            all_files.extend(directory.glob(f'*{ext}'))
            all_files.extend(directory.glob(f'*{ext.upper()}'))
        
        all_files = sorted(all_files)
        
        valid_images = []
        seen_images = set()
        
        for file_path in all_files:
            if file_path.name.startswith('._') or file_path.name.startswith('~'):
                continue
            
            img_identifier = file_path.stem.replace('._', '').replace('.', '')
            
            if img_identifier in seen_images:
                continue
            
            try:
                with Image.open(file_path) as img:
                    img.verify()
                valid_images.append(file_path)
                seen_images.add(img_identifier)
            except Exception:
                continue
        
        return valid_images
    
    def copy_images_to_folder(self, images: List[Path], output_folder: Path, collage_num: int):
        """Copy the images used in the collage to a corresponding folder"""
        if not images:
            return
        
        # Fixed grid: 5 columns, 4 rows
        cols, rows = 5, 4
        total_cells = cols * rows
        
        # Calculate which images to use for this collage
        start_idx = (collage_num - 1) * total_cells
        end_idx = start_idx + total_cells
        
        if start_idx >= len(images):
            return
        
        images_to_use = images[start_idx:end_idx]
        
        # If we don't have enough images, pad with images from the beginning
        if len(images_to_use) < total_cells:
            needed = total_cells - len(images_to_use)
            images_to_use.extend(images[:needed])
        
        # Create the images output folder
        output_folder.mkdir(parents=True, exist_ok=True)
        
        print(f"    Copying {len(images_to_use)} images to: {output_folder.relative_to(self.images_output_dir)}")
        
        # Copy each image with numbered filename
        for idx, img_path in enumerate(images_to_use):
            try:
                # Create numbered filename (01.jpg, 02.jpg, etc.)
                new_filename = f"{idx+1:02d}{img_path.suffix}"
                new_path = output_folder / new_filename
                
                # Copy the image file
                shutil.copy2(img_path, new_path)
                print(f"      Copied: {img_path.name} -> {new_filename}")
                
            except Exception as e:
                print(f"      Failed to copy {img_path.name}: {e}")
    
    def create_fixed_grid_collage(self, images: List[Path], output_path: Path, collage_num: int = 1, folder_path: Path = None) -> bool:
        """Create a collage with fixed 5x4 grid and black background"""
        if not images:
            return False
        
        # Fixed grid: 5 columns, 4 rows
        cols, rows = 5, 4
        total_cells = cols * rows
        
        # Calculate which images to use for this collage
        start_idx = (collage_num - 1) * total_cells
        end_idx = start_idx + total_cells
        
        if start_idx >= len(images):
            return False
        
        images_to_use = images[start_idx:end_idx]
        
        # If we don't have enough images, pad with images from the beginning
        if len(images_to_use) < total_cells:
            needed = total_cells - len(images_to_use)
            images_to_use.extend(images[:needed])
        
        # Calculate cell size with padding
        usable_width = self.collage_size[0] - (self.padding * (cols + 1))
        usable_height = self.collage_size[1] - (self.padding * (rows + 1))
        cell_width = usable_width // cols
        cell_height = usable_height // rows
        
        # Create black canvas
        canvas = Image.new('RGB', self.collage_size, (0, 0, 0))
        
        images_placed = 0
        
        for idx, img_path in enumerate(images_to_use):
            try:
                with Image.open(img_path) as img:
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    img_resized = img.copy()
                    img_resized.thumbnail((cell_width, cell_height), Image.Resampling.LANCZOS)
                    
                    # Calculate position in 5x4 grid
                    row = idx // cols
                    col = idx % cols
                    
                    x_pos = self.padding + col * (cell_width + self.padding)
                    y_pos = self.padding + row * (cell_height + self.padding)
                    
                    # Center in cell
                    x_offset = (cell_width - img_resized.width) // 2
                    y_offset = (cell_height - img_resized.height) // 2
                    
                    final_x = x_pos + x_offset
                    final_y = y_pos + y_offset
                    
                    canvas.paste(img_resized, (final_x, final_y))
                    images_placed += 1
                    
            except Exception:
                continue
        
        if images_placed > 0:
            canvas.save(output_path, quality=95, optimize=True)
            
            # Copy source images to corresponding folder
            if folder_path:
                relative_path = folder_path.relative_to(self.input_dir)
                images_output_folder = self.images_output_dir / relative_path / f"collage_{collage_num:03d}"
                self.copy_images_to_folder(images, images_output_folder, collage_num)
            
            return True
        
        return False
    
    def analyze_folder_structure(self) -> None:
        """Analyze the complete folder structure"""
        print(f"Analyzing folder structure in: {self.input_dir}")
        
        total_images_found = 0
        
        for root, dirs, files in os.walk(self.input_dir):
            root_path = Path(root)
            
            if self.output_dir in root_path.parents or root_path == self.output_dir:
                continue
            
            images = self.find_valid_images(root_path)
            
            if images:
                try:
                    depth = len(root_path.relative_to(self.input_dir).parts)
                except ValueError:
                    depth = 0
                
                self.folder_levels[depth].append(root_path)
                self.folder_images[root_path] = images
                total_images_found += len(images)
                
                print(f"  Level {depth}: {len(images)} images in {root_path.relative_to(self.input_dir)}")
        
        print(f"\n📊 Total: {total_images_found} images across {sum(len(folders) for folders in self.folder_levels.values())} folders")
    
    def process_folders(self) -> None:
        """Process all folders and create fixed grid collages + copy images"""
        if not self.folder_levels:
            print("No folders with images found!")
            return
        
        max_depth = max(self.folder_levels.keys())
        
        print(f"\nProcessing {len(self.folder_levels)} levels")
        
        # Process all folders (not just leaves)
        for depth in range(max_depth, -1, -1):
            if depth not in self.folder_levels:
                continue
            
            folders = self.folder_levels[depth]
            print(f"\n--- Processing level {depth} ({len(folders)} folders) ---")
            
            for folder_path in folders:
                images = self.folder_images.get(folder_path, [])
                if not images:
                    continue
                
                # Create output path that mirrors input structure
                relative_path = folder_path.relative_to(self.input_dir)
                output_folder_path = self.output_dir / relative_path
                output_folder_path.mkdir(parents=True, exist_ok=True)
                
                print(f"  Processing: {relative_path} ({len(images)} images)")
                
                # Calculate how many collages to create (20 images each)
                max_possible_collages = len(images) // 20
                if len(images) % 20 > 0:
                    max_possible_collages += 1
                
                collages_to_create = min(self.collages_per_folder, max_possible_collages)
                
                # Create collages for this folder
                for collage_num in range(1, collages_to_create + 1):
                    output_filename = f"collage_{collage_num:03d}.png"
                    output_path = output_folder_path / output_filename
                    
                    if self.create_fixed_grid_collage(images, output_path, collage_num, folder_path):
                        print(f"    Created: {output_filename}")
    
    def run(self) -> None:
        """Main execution"""
        print("=" * 60)
        print("Fixed Grid Collage Generator - 5x4 Grid + Image Copy")
        print("=" * 60)
        print(f"Input directory: {self.input_dir}")
        print(f"Collage output directory: {self.output_dir}")
        print(f"Image output directory: {self.images_output_dir}")
        print(f"Grid: 5 columns × 4 rows (20 images per collage)")
        print(f"Collages per folder: {self.collages_per_folder}")
        print(f"Collage size: {self.collage_size[0]}x{self.collage_size[1]}")
        print(f"Background: Black")
        print(f"Padding: {self.padding}px")
        print("=" * 60)
        
        self.analyze_folder_structure()
        self.process_folders()
        
        print("\n" + "=" * 60)
        print("Fixed grid collage generation complete!")
        print(f"Collages saved to: {self.output_dir}")
        print(f"Source images copied to: {self.images_output_dir}")
        print("=" * 60)


def create_fixed_grid_collages(input_folder: str, 
                              output_folder: str = None,
                              collages_per_folder: int = 20,
                              **kwargs):
    """
    Create fixed grid collages (5x4) with black background and copy source images.
    """
    generator = FixedGridCollageGenerator(
        input_dir=input_folder,
        output_dir=output_folder,
        collages_per_folder=collages_per_folder,
        **kwargs
    )
    generator.run()
    return generator


print("✅ Enhanced Fixed Grid Collage Generator Ready!")
print("\nFeatures:")
print("1. Fixed 5x4 grid (20 images per collage)")
print("2. Black background")
print("3. Organized folder structure")
print("4. Customizable collages per folder")
print("5. Creates TWO output folders:")
print("   - collages_fixed_grid/ : Contains the 5x4 collage images")
print("   - collage_images/ : Contains numbered source images used in each collage")

print("\nOutput Structure Example:")
print("""
collages_fixed_grid/
└── art_nouveau/
    ├── collage_001.png  (contains images 1-20)
    ├── collage_002.png  (contains images 21-40)
    └── ...

collage_images/
└── art_nouveau/
    ├── collage_001/
    │   ├── 01.jpg
    │   ├── 02.jpg
    │   ├── ...
    │   └── 20.jpg
    ├── collage_002/
    │   ├── 01.jpg
    │   ├── 02.jpg
    │   ├── ...
    │   └── 20.jpg
    └── ...
""")

print("\nUsage:")
print("""
create_fixed_grid_collages(
    input_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\artbench-10-imagefolder",
    output_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\collages_fixed",
    collages_per_folder=20,
    collage_size=(1920, 1080),
    padding=5
)
""")

✅ Enhanced Fixed Grid Collage Generator Ready!

Features:
1. Fixed 5x4 grid (20 images per collage)
2. Black background
3. Organized folder structure
4. Customizable collages per folder
5. Creates TWO output folders:
   - collages_fixed_grid/ : Contains the 5x4 collage images
   - collage_images/ : Contains numbered source images used in each collage

Output Structure Example:

collages_fixed_grid/
└── art_nouveau/
    ├── collage_001.png  (contains images 1-20)
    ├── collage_002.png  (contains images 21-40)
    └── ...

collage_images/
└── art_nouveau/
    ├── collage_001/
    │   ├── 01.jpg
    │   ├── 02.jpg
    │   ├── ...
    │   └── 20.jpg
    ├── collage_002/
    │   ├── 01.jpg
    │   ├── 02.jpg
    │   ├── ...
    │   └── 20.jpg
    └── ...


Usage:

create_fixed_grid_collages(
    input_folder="C:\\Users\\isabe\\Unity\\artbench-10-imagefolder",
    output_folder="C:\\Users\\isabe\\Unity\\collages_fixed",
    collages_per_folder=20,
    collage_size=(1920, 1080),
    padding

In [29]:
#!/usr/bin/env python3
"""
Fixed Hierarchical Collage Processor
Creates collages of collages from the output of fixed grid collages
"""

import os
import sys
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
from PIL import Image
import numpy as np


class FixedHierarchicalCollageProcessor:
    """Process collages hierarchically to create collages of collages."""
    
    def __init__(self, 
                 input_dir: str,
                 output_dir: Optional[str] = None,
                 collages_per_collage: int = 20,  # How many collages to combine
                 collage_size: Tuple[int, int] = (1920, 1080),
                 padding: int = 5):
        
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir) if output_dir else self.input_dir / 'hierarchical_collages_fixed'
        self.collages_per_collage = collages_per_collage
        self.collage_size = collage_size
        self.padding = padding
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def find_collage_files(self, directory: Path) -> List[Path]:
        """Find all PNG collage files in a directory"""
        collage_files = list(directory.glob('collage_*.png'))
        return sorted(collage_files)
    
    def create_collage_of_collages(self, collage_paths: List[Path], output_path: Path) -> bool:
        """Create a collage from existing collage images with black fill for empty cells"""
        if not collage_paths:
            print("    No collage paths provided")
            return False
        
        num_collages = len(collage_paths)
        print(f"    Creating hierarchical collage with {num_collages} collages")
        
        # Always use 5x4 grid for consistency (20 cells total)
        cols, rows = 5, 4
        total_cells = cols * rows
        
        # Calculate cell size
        usable_width = self.collage_size[0] - (self.padding * (cols + 1))
        usable_height = self.collage_size[1] - (self.padding * (rows + 1))
        cell_width = usable_width // cols
        cell_height = usable_height // rows
        
        print(f"    Cell size: {cell_width}x{cell_height}")
        
        # Create black canvas
        canvas = Image.new('RGB', self.collage_size, (0, 0, 0))
        
        collages_placed = 0
        
        # Place available collages
        for idx, collage_path in enumerate(collage_paths[:total_cells]):
            try:
                print(f"      Processing: {collage_path.name}")
                with Image.open(collage_path) as img:
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    img_resized = img.copy()
                    img_resized.thumbnail((cell_width, cell_height), Image.Resampling.LANCZOS)
                    
                    # Calculate position
                    row = idx // cols
                    col = idx % cols
                    
                    x_pos = self.padding + col * (cell_width + self.padding)
                    y_pos = self.padding + row * (cell_height + self.padding)
                    
                    # Center in cell
                    x_offset = (cell_width - img_resized.width) // 2
                    y_offset = (cell_height - img_resized.height) // 2
                    
                    final_x = x_pos + x_offset
                    final_y = y_pos + y_offset
                    
                    canvas.paste(img_resized, (final_x, final_y))
                    collages_placed += 1
                    print(f"      Placed at position ({col}, {row})")
                    
            except Exception as e:
                print(f"      Error processing {collage_path.name}: {e}")
                continue
        
        # Calculate empty cells for black fill
        empty_cells = total_cells - num_collages if num_collages < total_cells else 0
        
        if collages_placed > 0:
            canvas.save(output_path, quality=95, optimize=True)
            print(f"    ✅ Created hierarchical collage: {output_path.name}")
            print(f"    Collages placed: {collages_placed}, Empty cells: {empty_cells}")
            return True
        else:
            print(f"    ❌ Failed to create hierarchical collage: no collages placed")
            return False
    
    def get_folder_structure(self) -> Dict[int, List[Path]]:
        """Get all folders with collages, organized by depth"""
        print("Scanning folder structure...")
        
        collage_folders = []
        for root, dirs, files in os.walk(self.input_dir):
            root_path = Path(root)
            # Look for folders that contain collage files
            collage_files = self.find_collage_files(root_path)
            if collage_files:
                collage_folders.append(root_path)
                print(f"  Found {len(collage_files)} collages in: {root_path.relative_to(self.input_dir)}")
        
        if not collage_folders:
            print("No collage folders found!")
            return {}
        
        # Group by depth
        folder_levels = defaultdict(list)
        for folder in collage_folders:
            try:
                depth = len(folder.relative_to(self.input_dir).parts)
                folder_levels[depth].append(folder)
            except ValueError:
                continue
        
        print(f"\nFolder structure analysis:")
        for depth in sorted(folder_levels.keys()):
            print(f"  Level {depth}: {len(folder_levels[depth])} folders")
        
        return folder_levels
    
    def process_hierarchically(self) -> None:
        """Process folders hierarchically from deepest level to root"""
        folder_levels = self.get_folder_structure()
        
        if not folder_levels:
            print("No folder structure found!")
            return
        
        max_depth = max(folder_levels.keys())
        print(f"\nMaximum depth: {max_depth}")
        print("Starting hierarchical processing from deepest level...")
        
        # We need to track which folders we've processed to avoid duplicates
        processed_folders = set()
        
        # Process from deepest to shallowest
        for current_depth in range(max_depth, -1, -1):
            if current_depth not in folder_levels:
                print(f"Skipping level {current_depth} (no folders)")
                continue
            
            print(f"\n{'='*50}")
            print(f"PROCESSING LEVEL {current_depth}")
            print(f"{'='*50}")
            
            level_folders = folder_levels[current_depth]
            
            # Group folders by their parent
            parent_groups = defaultdict(list)
            for folder in level_folders:
                if folder in processed_folders:
                    continue
                    
                parent = folder.parent
                # Only process if parent is within our input directory structure
                if self.input_dir in parent.parents or parent == self.input_dir:
                    parent_groups[parent].append(folder)
                    processed_folders.add(folder)
            
            if not parent_groups:
                print("No parent groups found at this level")
                continue
            
            # Process each parent group
            for parent_path, child_folders in parent_groups.items():
                print(f"\n📁 Processing parent: {parent_path.relative_to(self.input_dir)}")
                print(f"   Child folders: {len(child_folders)}")
                
                # Collect all collages from all child folders
                all_collages = []
                for child_folder in child_folders:
                    collages = self.find_collage_files(child_folder)
                    all_collages.extend(collages)
                    print(f"   └─ {child_folder.name}: {len(collages)} collages")
                
                if not all_collages:
                    print("   ⚠️  No collages found in child folders!")
                    continue
                
                print(f"   Total collages collected: {len(all_collages)}")
                
                # Create output folder for parent (mirroring input structure)
                output_parent_path = self.output_dir / parent_path.relative_to(self.input_dir)
                output_parent_path.mkdir(parents=True, exist_ok=True)
                
                # Group collages for hierarchical processing
                collage_groups = [all_collages[i:i + self.collages_per_collage] 
                                for i in range(0, len(all_collages), self.collages_per_collage)]
                
                print(f"   Creating {len(collage_groups)} hierarchical collage(s)")
                
                # Create hierarchical collages
                created_count = 0
                for group_num, collage_group in enumerate(collage_groups, 1):
                    output_filename = f"hierarchical_collage_{group_num:03d}.png"
                    output_path = output_parent_path / output_filename
                    
                    if self.create_collage_of_collages(collage_group, output_path):
                        created_count += 1
                
                print(f"   ✅ Created {created_count} hierarchical collages in {output_parent_path.relative_to(self.output_dir)}")
        
        print(f"\n{'='*60}")
        print("✅ HIERARCHICAL PROCESSING COMPLETE!")
        print(f"{'='*60}")
        print(f"Final collages saved to: {self.output_dir}")
    
    def run(self) -> None:
        """Main execution"""
        print("=" * 60)
        print("Fixed Hierarchical Collage Processor")
        print("=" * 60)
        print(f"Input directory: {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Collages per hierarchical collage: {self.collages_per_collage}")
        print(f"Collage size: {self.collage_size[0]}x{self.collage_size[1]}")
        print(f"Grid: 5x4 (20 cells)")
        print(f"Background: Black")
        print(f"Padding: {self.padding}px")
        print("=" * 60)
        
        self.process_hierarchically()


def create_fixed_hierarchical_collages(input_folder: str,
                                      output_folder: str = None,
                                      collages_per_collage: int = 20,
                                      **kwargs):
    """
    Create hierarchical collages from the output of fixed grid collages.
    
    Args:
        input_folder: Path to the collages_fixed directory
        output_folder: Where to save hierarchical collages
        collages_per_collage: How many collages to combine into one
        **kwargs: collage_size, padding
    """
    processor = FixedHierarchicalCollageProcessor(
        input_dir=input_folder,
        output_dir=output_folder,
        collages_per_collage=collages_per_collage,
        **kwargs
    )
    processor.run()
    return processor


print("✅ Fixed Hierarchical Collage Processor Ready!")
print("\nUsage:")
print("""
# Process the output from fixed grid collages
create_fixed_hierarchical_collages(
    input_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\collages_fixed",
    output_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\hierarchical_collages_fixed",
    collages_per_collage=20,
    collage_size=(1920, 1080),
    padding=5
)
""")

✅ Fixed Hierarchical Collage Processor Ready!

Usage:

# Process the output from fixed grid collages
create_fixed_hierarchical_collages(
    input_folder="C:\\Users\\isabe\\Unity\\collages_fixed",
    output_folder="C:\\Users\\isabe\\Unity\\hierarchical_collages_fixed",
    collages_per_collage=20,
    collage_size=(1920, 1080),
    padding=5
)



In [7]:
# Step 1: Create fixed grid collages
create_fixed_grid_collages(
    input_folder="C:\\Users\\isabe\\Unity\\SpatialVR_data",
    output_folder="C:\\Users\\isabe\\Unity\\collages_fixed2",
    collages_per_folder=20,
    collage_size=(1920, 1080),
    padding=5
)

Fixed Grid Collage Generator - 5x4 Grid + Image Copy
Input directory: C:\Users\isabe\Unity\SpatialVR_data
Collage output directory: C:\Users\isabe\Unity\collages_fixed2
Image output directory: C:\Users\isabe\Unity\collage_images
Grid: 5 columns × 4 rows (20 images per collage)
Collages per folder: 20
Collage size: 1920x1080
Background: Black
Padding: 5px
Analyzing folder structure in: C:\Users\isabe\Unity\SpatialVR_data
  Level 1: 6000 images in art_nouveau
  Level 1: 6000 images in baroque
  Level 1: 6000 images in expressionism
  Level 1: 6000 images in impressionism
  Level 1: 6000 images in post_impressionism
  Level 1: 6000 images in realism
  Level 1: 6000 images in renaissance
  Level 1: 6000 images in romanticism
  Level 1: 6000 images in surrealism
  Level 1: 6000 images in ukiyo_e

📊 Total: 60000 images across 10 folders

Processing 1 levels

--- Processing level 1 (10 folders) ---
  Processing: art_nouveau (6000 images)
    Copying 20 images to: art_nouveau\collage_001
     

In [30]:
# Run only the hierarchical processor on your existing collages_fixed folder
create_fixed_hierarchical_collages(
    input_folder="C:\\Users\\isabe\\Unity\\collages_fixed",
    output_folder="C:\\Users\\isabe\\Unity\\hierarchical_collages_fixed",
    collages_per_collage=20,
    collage_size=(1920, 1080),
    padding=5
)

Fixed Hierarchical Collage Processor
Input directory: C:\Users\isabe\Unity\collages_fixed
Output directory: C:\Users\isabe\Unity\hierarchical_collages_fixed
Collages per hierarchical collage: 20
Collage size: 1920x1080
Grid: 5x4 (20 cells)
Background: Black
Padding: 5px
Scanning folder structure...
  Found 20 collages in: art_nouveau
  Found 20 collages in: baroque
  Found 20 collages in: expressionism
  Found 20 collages in: impressionism
  Found 20 collages in: post_impressionism
  Found 20 collages in: realism
  Found 20 collages in: renaissance
  Found 20 collages in: romanticism
  Found 20 collages in: surrealism
  Found 20 collages in: ukiyo_e

Folder structure analysis:
  Level 1: 10 folders

Maximum depth: 1
Starting hierarchical processing from deepest level...

PROCESSING LEVEL 1

📁 Processing parent: .
   Child folders: 10
   └─ art_nouveau: 20 collages
   └─ baroque: 20 collages
   └─ expressionism: 20 collages
   └─ impressionism: 20 collages
   └─ post_impressionism: 20 c

In [5]:
#!/usr/bin/env python3
"""
Image Compressor for Collage Output
Compresses collage images to be no more than 50 KB each
"""

import os
from pathlib import Path
from PIL import Image
import shutil


class CollageImageCompressor:
    """Compress collage images to reduce file size."""
    
    def __init__(self, input_dir: str, output_dir: Optional[str] = None, max_size_kb: int = 50):
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir) if output_dir else self.input_dir.parent / f"{self.input_dir.name}_compressed"
        self.max_size_kb = max_size_kb
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def compress_image(self, input_path: Path, output_path: Path) -> bool:
        """Compress a single image to be under max_size_kb"""
        try:
            with Image.open(input_path) as img:
                # Convert to RGB if necessary (JPEG doesn't support RGBA)
                if img.mode in ('RGBA', 'P'):
                    img = img.convert('RGB')
                
                # Start with high quality and reduce until under size limit
                quality = 95
                min_quality = 10
                
                while quality >= min_quality:
                    # Save with current quality setting
                    img.save(output_path, 'JPEG', quality=quality, optimize=True)
                    
                    # Check file size
                    file_size_kb = output_path.stat().st_size / 1024
                    
                    if file_size_kb <= self.max_size_kb:
                        print(f"    ✓ Compressed: {file_size_kb:.1f} KB (quality: {quality})")
                        return True
                    
                    # Reduce quality for next attempt
                    quality -= 15
                
                # If we still can't get under the limit, use the smallest we achieved
                final_size_kb = output_path.stat().st_size / 1024
                print(f"    ⚠️  Minimum: {final_size_kb:.1f} KB (could not reach {self.max_size_kb} KB)")
                return True
                
        except Exception as e:
            print(f"    ✗ Error compressing {input_path.name}: {e}")
            return False
    
    def compress_folder_structure(self) -> None:
        """Compress all images while maintaining folder structure"""
        print(f"Compressing images in: {self.input_dir}")
        print(f"Target max size: {self.max_size_kb} KB per image")
        print(f"Output directory: {self.output_dir}")
        
        total_images = 0
        compressed_images = 0
        
        # Walk through the entire directory structure
        for root, dirs, files in os.walk(self.input_dir):
            root_path = Path(root)
            
            # Calculate relative path for output directory
            relative_path = root_path.relative_to(self.input_dir)
            output_folder_path = self.output_dir / relative_path
            output_folder_path.mkdir(parents=True, exist_ok=True)
            
            # Process image files in this folder
            image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            if image_files:
                print(f"\n📁 Processing: {relative_path or 'root'}")
                
                for file_name in image_files:
                    total_images += 1
                    input_file_path = root_path / file_name
                    
                    # Create output filename (convert to JPEG for better compression)
                    output_file_name = file_name
                    if output_file_name.lower().endswith('.png'):
                        output_file_name = output_file_name[:-4] + '.jpg'
                    
                    output_file_path = output_folder_path / output_file_name
                    
                    # Get original file size
                    original_size_kb = input_file_path.stat().st_size / 1024
                    print(f"  {file_name}: {original_size_kb:.1f} KB → ", end="")
                    
                    # Compress the image
                    if self.compress_image(input_file_path, output_file_path):
                        compressed_images += 1
                    else:
                        # If compression fails, copy the original
                        shutil.copy2(input_file_path, output_file_path)
                        print(f"    Copied original: {original_size_kb:.1f} KB")
        
        print(f"\n{'='*60}")
        print("COMPRESSION SUMMARY")
        print(f"{'='*60}")
        print(f"Total images processed: {total_images}")
        print(f"Successfully compressed: {compressed_images}")
        print(f"Output directory: {self.output_dir}")
        print(f"Max target size: {self.max_size_kb} KB")
        print(f"{'='*60}")
    
    def run(self) -> None:
        """Main execution"""
        print("=" * 60)
        print("Collage Image Compressor")
        print("=" * 60)
        print(f"Input directory: {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Max file size: {self.max_size_kb} KB")
        print("=" * 60)
        
        self.compress_folder_structure()


def compress_collage_images(input_folder: str, 
                           output_folder: str = None, 
                           max_size_kb: int = 50):
    """
    Compress collage images to be no more than 50 KB each.
    
    Args:
        input_folder: Path to collages_fixed_grid folder
        output_folder: Where to save compressed images (optional)
        max_size_kb: Maximum file size in KB (default: 50)
    """
    compressor = CollageImageCompressor(
        input_dir=input_folder,
        output_dir=output_folder,
        max_size_kb=max_size_kb
    )
    compressor.run()
    return compressor


# Function to analyze current file sizes
def analyze_folder_sizes(folder_path: str):
    """Analyze current file sizes in a folder"""
    folder = Path(folder_path)
    total_size = 0
    file_count = 0
    sizes = []
    
    print(f"Analyzing file sizes in: {folder}")
    
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                file_path = Path(root) / file
                size_kb = file_path.stat().st_size / 1024
                sizes.append(size_kb)
                total_size += size_kb
                file_count += 1
    
    if sizes:
        avg_size = total_size / file_count
        max_size = max(sizes)
        min_size = min(sizes)
        
        print(f"📊 File Size Analysis:")
        print(f"   Total files: {file_count}")
        print(f"   Average size: {avg_size:.1f} KB")
        print(f"   Largest file: {max_size:.1f} KB")
        print(f"   Smallest file: {min_size:.1f} KB")
        print(f"   Total size: {total_size/1024:.2f} MB")
        
        # Count files over 50KB
        over_limit = sum(1 for size in sizes if size > 50)
        print(f"   Files over 50KB: {over_limit} ({over_limit/file_count*100:.1f}%)")
    else:
        print("No image files found!")


print("✅ Collage Image Compressor Ready!")
print("\nFeatures:")
print("1. Compresses images to ≤ 50 KB each")
print("2. Maintains original folder structure")
print("3. Converts PNG to JPEG for better compression")
print("4. Progressive quality reduction to hit target size")
print("5. Preserves folder hierarchy")

print("\nUsage Examples:")

# First, let's analyze the current sizes
print("""
# Analyze current file sizes
analyze_folder_sizes("C:\\\\Users\\\\isabe\\\\Unity\\\\collages_fixed")

# Compress all images to 50KB max
compress_collage_images(
    input_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\collages_fixed",
    output_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\collages_compressed",
    max_size_kb=50
)

# Compress with different size limit
compress_collage_images(
    input_folder="C:\\\\Users\\\\isabe\\\\Unity\\\\collages_fixed", 
    max_size_kb=30  # Even smaller files
)
""")

✅ Collage Image Compressor Ready!

Features:
1. Compresses images to ≤ 50 KB each
2. Maintains original folder structure
3. Converts PNG to JPEG for better compression
4. Progressive quality reduction to hit target size
5. Preserves folder hierarchy

Usage Examples:

# Analyze current file sizes
analyze_folder_sizes("C:\\Users\\isabe\\Unity\\collages_fixed")

# Compress all images to 50KB max
compress_collage_images(
    input_folder="C:\\Users\\isabe\\Unity\\collages_fixed",
    output_folder="C:\\Users\\isabe\\Unity\\collages_compressed",
    max_size_kb=50
)

# Compress with different size limit
compress_collage_images(
    input_folder="C:\\Users\\isabe\\Unity\\collages_fixed", 
    max_size_kb=30  # Even smaller files
)



In [6]:
# Analyze current file sizes in your collages_fixed folder
analyze_folder_sizes("C:\\Users\\isabe\\Unity\\collages_fixed")

Analyzing file sizes in: C:\Users\isabe\Unity\collages_fixed
📊 File Size Analysis:
   Total files: 200
   Average size: 2258.9 KB
   Largest file: 2804.5 KB
   Smallest file: 1309.1 KB
   Total size: 441.19 MB
   Files over 50KB: 200 (100.0%)


In [7]:
# Compress all images to be no more than 50 KB each
compress_collage_images(
    input_folder="C:\\Users\\isabe\\Unity\\collages_fixed",
    output_folder="C:\\Users\\isabe\\Unity\\collages_compressed",
    max_size_kb=50
)

Collage Image Compressor
Input directory: C:\Users\isabe\Unity\collages_fixed
Output directory: C:\Users\isabe\Unity\collages_compressed
Max file size: 50 KB
Compressing images in: C:\Users\isabe\Unity\collages_fixed
Target max size: 50 KB per image
Output directory: C:\Users\isabe\Unity\collages_compressed

📁 Processing: art_nouveau
  collage_001.png: 2548.7 KB →     ⚠️  Minimum: 121.3 KB (could not reach 50 KB)
  collage_002.png: 2498.2 KB →     ⚠️  Minimum: 124.3 KB (could not reach 50 KB)
  collage_003.png: 2576.6 KB →     ⚠️  Minimum: 125.6 KB (could not reach 50 KB)
  collage_004.png: 2333.3 KB →     ⚠️  Minimum: 109.9 KB (could not reach 50 KB)
  collage_005.png: 2283.7 KB →     ⚠️  Minimum: 99.3 KB (could not reach 50 KB)
  collage_006.png: 2198.4 KB →     ⚠️  Minimum: 89.9 KB (could not reach 50 KB)
  collage_007.png: 2261.8 KB →     ⚠️  Minimum: 99.7 KB (could not reach 50 KB)
  collage_008.png: 2367.7 KB →     ⚠️  Minimum: 112.0 KB (could not reach 50 KB)
  collage_009.png: 